In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path for local imports
project_root = Path('..').resolve()  # Assuming notebook is in 'notebooks/'
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config_loader import load_config, get_seed, get_path
from src.data_process import (
    load_and_merge_data,
    apply_extinction_correction,
    extract_features_with_gp
)

# Load Configuration
config = load_config()

# Set LEGACY Seed for 100% Reproduction
SEED = get_seed(legacy=True)
print(f"🔹 Using Legacy Seed: {SEED}")

In [ ]:
import numpy as np
import pandas as pd
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

class Config:
    # Determine Input Root based on Mode
    if config.get('mode') == 'train':  # Kaggle/Cloud Mode
        INPUT_ROOT = '/kaggle/input/raw-mallorn' 
        print("Running in KAGGLE/TRAIN mode")
    else:  # Local Mode
        INPUT_ROOT = str(get_path('data_raw'))
        print(f"Running in LOCAL mode. Root: {INPUT_ROOT}")

    OUTPUT_DIR = str(get_path('data_processed'))
    OUTPUT_FILE_TRAIN = os.path.join(OUTPUT_DIR, 'mallorn_train_features.parquet')
    OUTPUT_FILE_TEST = os.path.join(OUTPUT_DIR, 'mallorn_test_features.parquet')
    
    # Augmentation Settings
    N_AUGMENTATIONS = 2 
    PROCESS_GP = True
    
    # Ensure output directory exists
    os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Load Metadata (Logs)
print("Loading Metadata...")
train_log = pd.read_csv(os.path.join(Config.INPUT_ROOT, 'train_log.csv'))
test_log = pd.read_csv(os.path.join(Config.INPUT_ROOT, 'test_log.csv'))

# Load Lightcurves using Legacy Logic
print("Loading Train Lightcurves...")
train_lc = load_and_merge_data(train_log, Config.INPUT_ROOT, is_train=True)

print("Loading Test Lightcurves...")
test_lc = load_and_merge_data(test_log, Config.INPUT_ROOT, is_train=False)

In [ ]:
# Apply Extinction Correction
train_lc = apply_extinction_correction(train_lc, train_log)
test_lc = apply_extinction_correction(test_lc, test_log)

In [ ]:
# Extract Features using Legacy GP Logic
print("\nExtracting Train Features...")
train_feats = extract_features_with_gp(train_lc, train_log, seed=SEED, use_gp=Config.PROCESS_GP)

print("\nExtracting Test Features...")
test_feats = extract_features_with_gp(test_lc, test_log, seed=SEED, use_gp=Config.PROCESS_GP)

In [ ]:
print(f"\n💾 Saving processed features to {Config.OUTPUT_DIR}...")

train_feats.to_parquet(Config.OUTPUT_FILE_TRAIN, index=False)
test_feats.to_parquet(Config.OUTPUT_FILE_TEST, index=False)

print("Done. Reproducibility ensured.")